# GenAI Orchestration and Agent Patterns

This course covers how LLM agents reason, plan, collaborate, and recover from failure. It progresses from single-agent multi-step reasoning through multi-agent architectures, tool integration, error handling, and production orchestration best practices.

## Table of Contents

- [Module 01: Multi-step Reasoning and Planning for LLM Agents](#module-01-multi-step-reasoning-and-planning-for-llm-agents)
- [Module 02: Multi-agent Architectures and Collaboration](#module-02-multi-agent-architectures-and-collaboration)
- [Module 03: Tool Integration and Chaining](#module-03-tool-integration-and-chaining)
- [Module 04: Error Handling and Recovery in Agent Systems](#module-04-error-handling-and-recovery-in-agent-systems)
- [Module 05: Agent Orchestration and Best Practices](#module-05-agent-orchestration-and-best-practices)

## Module 01: Multi-step Reasoning and Planning for LLM Agents

LLM agents that handle complex tasks cannot rely on single-shot responses — they must plan, act, and reflect across multiple steps. This module introduces the core cognitive phases, reasoning architectures (ReAct, Chain-of-thought, Tree-of-thought), planning systems, and verification techniques that enable reliable multi-step agent behavior.

### Introduction to Multi-step Reasoning

Single-step reasoning breaks down on complex tasks because it cannot account for dependencies, intermediate states, or long-term goals. Without planning or reflection, agents are unable to detect errors, adapt to new information, or correct flawed assumptions.

#### Core Phases of Multi-step Reasoning

| Phase | Role |
|---|---|
| **Planning** | Decompose the goal into ordered subtasks |
| **Execution** | Carry out each subtask, invoking tools as needed |
| **Reflection** | Evaluate intermediate results and decide whether to revise |

### Cognitive Architectures

#### ReAct: Reason + Act

ReAct interleaves explicit reasoning with concrete actions taken in the environment to solve tasks step by step. Reasoning guides what to do next, while actions — tool calls, API requests, searches, function execution — provide real-world feedback that informs the next reasoning step.

**When to use ReAct:**
- Tasks requiring frequent environment interaction
- Where reasoning and action are tightly interleaved
- Search, data retrieval, and orchestration workflows

#### Chain-of-thought: Sequential Reasoning

Chain-of-thought (CoT) enables agents to solve complex problems by generating a sequential series of intermediate reasoning steps before producing a final answer. Rather than jumping directly to a conclusion, the agent works through the logic step by step, making its reasoning transparent and correctable.

**When to use Chain-of-thought:**
- **Stepwise reasoning** — problems that decompose naturally into ordered steps
- **Explicit logic** — when each inference must be visible and auditable
- **Deterministic tasks** — arithmetic, rule application, structured deduction
- **Latency trade-offs** — CoT adds token cost but improves correctness on hard problems

#### Tree-of-thought: Parallel Path Exploration

Tree-of-thought (ToT) reasoning allows the agent to explore multiple possible reasoning paths in parallel, evaluate intermediate outcomes, and select the most promising solution before committing to a final answer. Unlike CoT's single linear chain, ToT branches at decision points and prunes unpromising paths.

**When to use Tree-of-thought:**
- **Complex problems** — where the optimal solution path is not obvious upfront
- **Open-ended tasks** — creative writing, multi-step planning with many valid routes
- **Strategic planning** — when intermediate state evaluation matters
- **High overhead** — ToT is computationally expensive; use only when correctness justifies the cost

### Planning Systems

Planning systems enable agents to decompose complex goals into structured, multi-step execution plans. They allow agents to reason about task order, dependencies, and adjustments based on intermediate results — transforming a vague objective into a concrete, executable sequence.

**Why planning is necessary:**
- Single-step reasoning fails on complex tasks
- Large objectives need structured decomposition
- Planning improves reliability and consistency
- Agents must reason about task dependencies
- Plans must adapt during execution

#### Implementing Planning Systems

A complete planning system involves six stages:

1. **Goal definition** — clarify and bound the objective
2. **Task decomposition** — break the goal into atomic subtasks
3. **Plan generation** — sequence subtasks with dependencies and decision points
4. **Plan execution** — invoke tools and agents per the plan
5. **Outcome evaluation** — assess each step's result
6. **Plan adjustment** — revise remaining steps based on findings

#### Generating Execution Plans

- Convert subtasks into a structured execution plan
- Define sequencing, dependencies, and decision points
- Represent plans as lists, graphs, or state machines
- Use LLMs to generate plans from natural language goals

#### Adapting Plans Using Intermediate Results

Plans are not static — effective agents update them as execution proceeds:

- Evaluate intermediate outputs against expectations
- Detect errors or gaps in completed steps
- Modify or regenerate pending subtasks
- Request additional tools or reasoning when stuck

### Reasoning Loops and Reflection

#### What Are Reasoning Loops?

Reasoning loops are iterative cycles where an agent reasons, acts, and evaluates progress toward a goal. They enable agents to verify intermediate results, detect errors, and decide whether further reasoning or tool use is needed.

**Key elements of an effective reasoning loop:**
- Track goals, state, and progress across iterations
- Verify intermediate results before proceeding
- Decide on the next action based on current state
- Define clear stopping conditions to prevent infinite loops

#### Verifying Intermediate Steps

> Verification prevents small reasoning errors from propagating and derailing the final outcome in multi-step tasks.

Techniques for verifying intermediate steps:
- Re-evaluate conclusions using alternative prompts
- Check outputs against constraints, rules, or expected formats
- Use self-consistency: compare multiple independent reasoning paths
- Insert verification checkpoints after critical reasoning steps
- Decide whether to proceed, revise, or re-plan based on outcomes

#### Verification Tools in Reasoning Loops

| Tool Type | Purpose |
|---|---|
| **Deterministic validation tools** | Assert invariants, type checks, schema validation |
| **External computation and execution tools** | Run code, calculate, simulate |
| **Retrieval and grounding tools** | Check facts against authoritative sources |
| **LLM-based verification tools** | Use a second LLM call to critique the output |
| **Monitoring and observability tools** | Log traces, detect anomalies, alert on failure |

### Hard-coded vs. Emergent Reasoning

Hard-coded reasoning uses predefined rules and control flows that enforce predictable, reliable reasoning at the cost of flexibility. Emergent reasoning is LLM-driven: it adapts dynamically to novel situations but introduces variability, cost, and less predictability.

| Dimension | Hard-coded | Emergent |
|---|---|---|
| **Logic source** | Predefined rules & control flow | LLM reasoning at runtime |
| **Adaptability** | Limited as tasks change | Adapts to novel & ambiguous tasks |
| **Reliability** | Higher — consistent outcomes | Higher variability |
| **Cost** | Predictable and controlled | Higher due to longer reasoning chains |
| **Latency** | Lower and stable | Higher and variable |
| **Flexibility** | Low | High |

**Guidance for choosing:**
- Use **hard-coded** for high-risk or regulated tasks and predictable, repeatable workflows
- Use **emergent** for open-ended or creative tasks and novel or dynamic situations
- **Combine both** to balance reliability and flexibility — hard-code the skeleton, use LLM reasoning for ambiguous branches

### Demo: Plan–Execute–Reflect Agent

This demo implements the three-phase agent loop: the agent first generates a plan for a task, then executes the plan step by step, then reflects on whether the result is correct and proposes corrections if needed.

See [`01/demos.ipynb`](01/demos.ipynb) for the full notebook.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

task = "Calculate the average of the numbers 10, 20, and 40, and explain the result."

# --- PLAN phase ---
plan_prompt = f"""You are an AI agent.\n\nTask: {task}\n\nFirst, create a simple step-by-step plan. Do NOT execute the steps yet."""
plan = client.responses.create(model="gpt-4.1-mini", input=plan_prompt).output_text
print("PLAN:\n", plan)

# --- EXECUTE phase ---
execute_prompt = f"""You previously created this plan:\n\n{plan}\n\nNow execute the plan step by step and produce the result."""
execution_result = client.responses.create(model="gpt-4.1-mini", input=execute_prompt).output_text
print("\nEXECUTION RESULT:\n", execution_result)

# --- REFLECT phase ---
reflect_prompt = f"""Here is the task:\n{task}\n\nHere is the result produced:\n{execution_result}\n\nReflect on the result:\n- Is it correct?\n- If there is a mistake, explain it.\n- If needed, provide a corrected answer."""
reflection = client.responses.create(model="gpt-4.1-mini", input=reflect_prompt).output_text
print("\nREFLECTION:\n", reflection)

PLAN:
 Step-by-step plan:

1. Identify the numbers to be averaged: 10, 20, and 40.
2. Add these numbers together to get the total sum.
3. Count the total number of numbers.
4. Divide the total sum by the number of numbers to find the average.
5. Explain the result in simple terms.

EXECUTION RESULT:
 Let's execute the plan step by step:

1. Identify the numbers to be averaged: 10, 20, and 40.
2. Add these numbers together to get the total sum:  
   \(10 + 20 + 40 = 70\).
3. Count the total number of numbers:  
   There are 3 numbers.
4. Divide the total sum by the number of numbers to find the average:  
   \(\frac{70}{3} \approx 23.33\).
5. Explain the result in simple terms:  
   The average of the numbers 10, 20, and 40 is approximately 23.33, which means if you split the total of these numbers equally into 3 parts, each part would be about 23.33.

REFLECTION:
 Let's reflect on the result step by step:

1. The numbers identified are correct: 10, 20, and 40.
2. The sum of the numbers

## Module 02: Multi-agent Architectures and Collaboration

Multi-agent systems coordinate multiple autonomous agents to tackle tasks that exceed a single agent's scope, capability, or context window. This module covers system types, collaboration patterns, role specialization, communication protocols, and conflict resolution.

### Multi-agent Systems

Multi-agent systems consist of multiple autonomous agents that interact, coordinate, or collaborate to achieve individual or shared goals. Agents can communicate, reason, and act independently while influencing the overall system behavior.

#### Types of Multi-agent Systems

| Type | Description |
|---|---|
| **Hierarchical** | Centralized control with delegated subagents; a top-level orchestrator breaks goals into subtasks |
| **Peer** | Autonomous agents collaborating without fixed hierarchy; tasks negotiated dynamically |
| **Specialized** | Expert agents each optimized for a distinct domain or capability |

#### Key Challenges

- **Goal alignment** — ensuring all agents work toward the same objective
- **Task duplication** — preventing multiple agents from doing the same work
- **Message ambiguity** — maintaining clear, unambiguous inter-agent communication
- **Communication latency** — managing delays in agent-to-agent message passing
- **Output conflicts** — resolving contradictory results from different agents
- **Context differences** — agents may have different views of shared state

### Multi-agent Collaboration Patterns

Collaboration patterns matter because they:
- Handle complex tasks that exceed single-agent scope
- Structure agent interactions clearly
- Reduce duplicated effort
- Manage coordination and conflict
- Improve overall system reliability

#### Hierarchical Task Delegation

A top-level agent breaks goals into subtasks and delegates each to a subagent matched by capability or expertise. Control flows downward while results flow upward. Agents operate without needing full system context — they only see their assigned work. This pattern improves scalability and reduces cognitive load per agent.

#### Debate and Consensus Mechanisms

Multiple agents independently propose solutions or perspectives, then converge through structured debate:
- **Structured debate** — agents argue for their position with evidence
- **Independent perspectives** — each agent reasons without seeing others' answers first
- **Shared consensus** — a coordinator synthesizes the proposals
- **Decision voting** — majority agreement determines the outcome
- **Bias reduction** — diverse agent viewpoints counteract individual model biases

#### Peer-to-peer Collaboration

Agents operate at the same level with no fixed hierarchy. Tasks are negotiated dynamically between agents. Information is shared directly via agent-to-agent messaging. Decisions emerge from collaboration rather than top-down delegation. This pattern increases flexibility and system resilience — no single point of failure.

### Agent Roles and Specialization

Agent roles define specific responsibilities, capabilities, and decision boundaries within a multi-agent system. Specialization improves efficiency and output quality by assigning tasks based on agent knowledge domains and strengths.

#### Assigning Tasks by Capability and Knowledge Domain

- Match tasks to agent capabilities and domain expertise
- Assign domain-specific tasks to expert agents
- Separate strategy (planning) from execution
- Use capability tags to enable dynamic task routing
- Define clear knowledge boundaries to reduce conflicts and redundancy

#### Balancing Roles

| Role Type | Characteristics |
|---|---|
| **Specialized** | Deep focus on a single capability or knowledge domain — high quality on narrow tasks |
| **Flexible** | Handles multiple tasks across domains — useful when task scope is unpredictable |

**Team composition** is the process of selecting and organizing agents with defined roles and capabilities to work together toward a shared goal. Effective strategies include:
- Task decomposition aligned with agent capabilities
- Balancing specialists with generalists
- Agent minimalism — use the fewest agents that can accomplish the goal
- Incremental scaling as complexity grows

### Communication Protocols

Agent-to-agent communication enables coordination, task delegation, and shared understanding across multiple agents. Clear communication patterns reduce ambiguity, improve reliability, and support scalable multi-agent systems.

#### Types of Agent-to-agent Communication

| Type | Description |
|---|---|
| **Send** | Directed message to a specific agent |
| **Request** | Ask another agent for a result or decision |
| **Broadcast** | Publish to all agents in the system |

#### Coordination and Task Delegation

- Agents align toward a shared objective via a coordination layer
- Tasks are assigned by capability and role
- High-level goals are decomposed into subtasks with clear expectations
- Status updates from subagents enable adaptive coordination

#### Decision and Conflict Resolution

When agents produce conflicting outputs:
- **Voting** — agents vote and the majority result wins
- **Majority agreement** — accept only outputs agreed upon by a quorum
- **Validation** — a dedicated validator agent checks outputs for correctness
- **Arbitration** — a coordinator agent makes the final call

#### Optimizing Multi-agent Communication

- Reduce duplication by routing only relevant context to each agent
- Use structured messages with consistent schemas
- Summarize results before passing them downstream
- Include contextual metadata to help receiving agents interpret messages

### Demo: Multi-agent Interaction

This demo shows task delegation across three specialized agents (Creative, Analytical, Marketing) followed by a Coordinator agent that synthesizes their outputs into a single consensus result.

See [`02/demos.ipynb`](02/demos.ipynb) for the full notebook.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def agent(agent_name, role, task):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"You are {agent_name}. Your role is: {role}."},
            {"role": "user", "content": task}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

task = "Suggest a short, catchy slogan for an AI healthcare startup."

responses = {
    "Creative Agent":    agent("Creative Agent",    "Come up with creative slogans", task),
    "Analytical Agent":  agent("Analytical Agent",  "Focus on clarity and trust", task),
    "Marketing Agent":   agent("Marketing Agent",   "Make it appealing to customers", task),
}
for name, resp in responses.items():
    print(f"{name}: {resp}\n")

# Coordinator synthesizes a consensus
summary = "The following agents proposed slogans:\n\n"
for name, resp in responses.items():
    summary += f"{name}: {resp}\n\n"
summary += "Based on consensus, choose the best single slogan."

final = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": summary}]
).choices[0].message.content
print("CONSENSUS SLOGAN:", final)

Creative Agent: "Empowering Health, One Byte at a Time!"

Analytical Agent: "Empowering Health, One Byte at a Time."

Marketing Agent: "Smart Care, Healthier You!"

CONSENSUS SLOGAN: Based on consensus, the best single slogan is: **"Smart Care, Healthier You!"** This slogan is catchy, impactful, and effectively conveys the message of promoting health in a positive way.


## Module 03: Tool Integration and Chaining

Tool-augmented agents extend their capabilities by invoking external tools, systems, and APIs. This module covers how agents discover and select tools, invoke them with structured inputs, interpret their outputs, chain multiple tools together, and maintain scalable tool registries.

### Tool-augmented Agents

Tool-augmented agents are AI agents that extend their capabilities by invoking external tools, systems, and APIs to perform actions beyond native model reasoning. They combine reasoning with action — enabling retrieval, execution, and interaction with real-world systems in a controlled and scalable way.

**Benefits of integrating external systems and APIs:**
- Enables actions beyond internal model capabilities
- Provides real-time access to external data
- Expands functionality without increasing model complexity
- Automates end-to-end workflows across systems
- Improves accuracy using authoritative external sources

**Key architectural considerations:**
- **Separation of concerns** — keep tool logic separate from agent reasoning
- **Tool abstraction** — uniform interface regardless of underlying implementation
- **Dynamic discovery** — agents find tools at runtime, not hard-coded
- **Context awareness** — tool selection informed by current task state
- **Output normalization** — consistent response format across all tools

### Tool Discovery and Selection

Tool discovery allows agents to understand what capabilities are available and how each tool can be used. Tool selection ensures the agent chooses the most appropriate tool based on task context and past execution history.

#### How Agents Identify Available Tools

- **Tool registry** — a central catalog of available tools with metadata
- **Standardized schemas** — machine-readable descriptions of inputs and outputs
- **Machine-readable descriptions** — plain-language summaries the LLM can reason over
- **Runtime querying** — agents query the registry dynamically at inference time
- **Dynamic discovery** — tools can be added or removed without changing agent code
- **Contextual filtering** — narrow the candidate set to tools relevant to the current task

#### Criteria for Selecting the Right Tool

- **Task relevance** — does the tool address what the task requires?
- **Data compatibility** — do the tool's input/output types match the current data?
- **Tool clarity** — is the tool's description unambiguous enough for reliable selection?
- **Reliability history** — has this tool succeeded on similar tasks before?
- **Performance latency** — is the tool fast enough for the current workflow?
- **Context alignment** — does the tool fit the broader task context?

#### Context and Past Performance

Agents select the most appropriate tool by combining real-time task context with historical performance data from similar interactions:
- Context defines task intent and constraints
- Input/output requirements guide selection
- Past success rates inform tool reliability
- Similar historical contexts improve accuracy of selection
- Combining context + performance enables adaptive tool selection that improves over time

### Tool Invocation and Result Interpretation

Agents call tools by creating structured requests that map reasoning outputs into validated, context-aware inputs compatible with the tool's interface.

#### How Agents Call Tools and Pass Inputs

- Agents invoke tools through standardized interfaces
- Tool calls use structured requests with typed parameters
- Inputs are derived from agent reasoning and prior context
- Context and prior outputs are passed alongside the request
- Inputs are serialized using formats like JSON

#### Handling Tool-specific Formats and Errors

Tool integration involves four concerns:

| Concern | What to address |
|---|---|
| **Different output formats and schemas** | Parse varied response structures into a uniform internal representation |
| **Interpreting tool responses** | Validate that the output matches expected types and semantics |
| **Error handling and failure strategies** | Retry, fallback, or surface errors to the agent for replanning |
| **Logging and observability** | Record inputs, outputs, and failures to enable learning from errors |

#### Interpreting Tool Outputs

- Tool outputs vary in structure and format across providers
- Parse structured outputs into internal representations
- Normalize responses for consistent downstream processing
- Extract only the information relevant to the current task
- Handle errors and unexpected tool responses gracefully

### Tool Chaining Strategies

Tool chaining connects multiple tool calls so that the output of one becomes the input of the next. Four primary strategies:

| Strategy | Description |
|---|---|
| **Sequential** | Tools execute one after another; each output feeds the next |
| **Parallel** | Multiple tools execute simultaneously on independent inputs |
| **Conditional** | A branching structure — which tool runs next depends on prior output |
| **Adaptive** | The agent selects the next tool dynamically based on accumulated results |

### Designing Tool Registries and Interfaces

#### Standardized Tool Registries and Interfaces

A well-designed registry enables dynamic tool addition without changing core agent logic, and ensures consistent discovery, invocation, and output handling across all tools.

**Principles for standardized interfaces:**
- Define a common tool contract that every tool must implement
- Specify consistent input schemas, output schemas, and error formats
- Use clear metadata: tool name, description, and supported actions
- Separate tool implementation from agent reasoning logic
- Tools can be added, removed, or updated without changing core code

#### Dynamic Tool Registration

- Add or remove tools at runtime via the central registry
- Use standardized tool metadata for discoverability
- Core agent logic remains unchanged when the tool set evolves
- Enables operational monitoring of tool usage and health

**Scalability and maintainability properties:**
- Loosely coupled — tools don't depend on each other
- Standardized interfaces — consistent contracts across all tools
- Dynamic registration — no hardcoded tool lists
- Logic separation — tool implementations isolated from agent code
- Operational monitoring — usage, errors, and latency tracked per tool

### Demo: Tool Chaining

This demo chains two tools sequentially: the first tool summarizes an input text, and the second generates quiz questions from that summary. The model uses function-calling to select tools and pass outputs between steps.

See [`03/demos.ipynb`](03/demos.ipynb) for the full notebook.

In [3]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Tool 1: Summarize text via LLM
def summarize_text(text: str) -> str:
    return client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Summarize the following text concisely."},
            {"role": "user", "content": text}
        ]
    ).choices[0].message.content

# Tool 2: Generate quiz questions from a summary
def generate_quiz(summary: str) -> str:
    return (
        "1. What is Agentic AI?\n"
        "2. Why is tool usage important for agents?\n"
        "3. Name one real-world use case of Agentic AI."
    )

tools = [
    {"type": "function", "function": {
        "name": "summarize_text",
        "description": "Summarize a given text",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}
    }},
    {"type": "function", "function": {
        "name": "generate_quiz",
        "description": "Generate quiz questions from a summary",
        "parameters": {"type": "object", "properties": {"summary": {"type": "string"}}, "required": ["summary"]}
    }}
]

user_prompt = """Summarize the following text and then create quiz questions from it.

Text:
Agentic AI systems can autonomously plan, reason, and act by using tools.
They are commonly used in workflows that require multi-step decision making."""

# Step 1: model selects the summarize tool
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_prompt}],
    tools=tools,
    tool_choice="auto"
)
message = response.choices[0].message
args = json.loads(message.tool_calls[0].function.arguments)

# Step 2: execute summarize tool
summary = summarize_text(**args)
print("Tool 1 Output (Summary):\n", summary)

# Step 3: chain into quiz tool
quiz = generate_quiz(summary=summary)
print("\nTool 2 Output (Quiz):\n", quiz)

Tool 1 Output (Summary):
 Agentic AI systems autonomously plan, reason, and act using tools, particularly in multi-step decision-making workflows.

Tool 2 Output (Quiz):
 1. What is Agentic AI?
2. Why is tool usage important for agents?
3. Name one real-world use case of Agentic AI.


## Module 04: Error Handling and Recovery in Agent Systems

Agent systems fail in predictable and unpredictable ways. This module covers the common failure modes — tool execution failures, reasoning errors, context overflow, and corrupted state — along with guardrails, logging strategies, and systematic debugging approaches.

### Common Failure Modes in LLM Agents

#### Tool Execution Failures

Tool calls fail due to issues such as invalid inputs, network errors, rate limits, or unexpected response formats. Agents should detect these failures, recover gracefully, and adapt by retrying or selecting alternative tools.

**Common execution failures:**
- **Network issues** — timeouts, connection refused, intermittent connectivity
- **Authentication failures** — expired tokens, missing credentials
- **Invalid inputs** — agent passes parameters the tool rejects
- **Rate limits** — too many calls in a time window
- **Schema mismatches** — tool returns an unexpected format
- **Partial results** — tool returns incomplete data without an error signal

#### Planning and Reasoning Errors

- Incorrect tool selection from incomplete task understanding
- Plans misaligned with real-world tool behavior
- Long tool chains amplify upstream reasoning errors
- Misinterpreted outputs propagate downstream failures
- Missing feedback prevents replanning when a step fails

#### Reasoning Breakdowns

| Failure | Description |
|---|---|
| **Infinite loops** | Repeated tool or reasoning execution without meaningful progress toward a goal |
| **Hallucinations** | Confident agent outputs not grounded in tool results or factual data |

### Context Management and Overflow Issues

#### Context Management

An agent's context includes the system prompt, conversation history, memory, and tool outputs. Effective context management ensures only the most relevant information is included at each step — poor context leads to incorrect reasoning or missed information.

- Context includes prompts, memory, tools, and prior results
- Context management ensures relevant inputs at each step
- Poor context selection causes incorrect reasoning
- Agents must actively control what gets included vs. retrieved

#### Overflow Issues

- Models have fixed token limits — context cannot grow unboundedly
- Exceeding limits causes inputs to be silently truncated
- Tool outputs can consume large portions of the context window
- Outdated memory included in context can cause conflicts with current state

**Mitigation strategies:** summarize prior history, use external memory with retrieval, filter tool outputs before passing them, implement context eviction policies.

#### Handling Corrupted and Inconsistent State

Corrupted state occurs when tools return invalid or partial outputs. Agents must validate, reset, or recover state to maintain reliable behavior:
- Validate tool outputs before integrating them into state
- Reset to a known-good checkpoint when state becomes inconsistent
- Log the corrupted state for post-hoc debugging
- Replan from the last valid step rather than continuing with bad data

### Safety Constraints and Guardrails

Safety constraints and guardrails are rules and limits that prevent agents from performing unsafe, unintended, or noncompliant actions while interacting with tools and external systems.

#### Types of Guardrails

| Guardrail Type | Purpose |
|---|---|
| **Safety guardrails** | Prevent harmful outputs or actions |
| **Input validation guardrails** | Reject malformed or unsafe inputs before tool invocation |
| **Output validation guardrails** | Verify that tool and agent outputs meet quality and safety criteria |
| **Tool usage guardrails** | Restrict which tools an agent can invoke in a given context |
| **Policy and compliance guardrails** | Enforce legal, regulatory, and organizational constraints |

**Safety and control guardrail categories:**
- **Resource and budget limits** — cap token spend, API call count, execution time
- **Action and decision restrictions** — prohibit irreversible or high-risk actions without confirmation
- **Avoiding harmful or unsafe behavior** — detect and block outputs that violate safety policies

#### Design Considerations for Guardrails

- Define clear behavioral boundaries for agent actions
- Enforce constraints at both the reasoning and execution layers
- Restrict tool access based on risk level and context
- Apply resource and budget limits per task or session
- Detect and stop unproductive agent loops automatically

### Logging and Debugging Strategies

#### Logging and Debugging

Effective logging captures agent execution data — reasoning traces, tool calls, inputs, outputs, and decisions — so that failures can be diagnosed and agent behavior improved over time.

#### Capturing Reasoning Traces

- Record intermediate reasoning steps in the agent's response
- Store prompts and all state transitions
- Correlate reasoning steps with their resulting tool calls
- Detect faulty assumptions early by reviewing traces
- Balance trace detail against cost — full traces are expensive at scale

#### Logging Tool Interactions and Results

- Log tool inputs, parameters, and metadata for each call
- Capture tool outputs, errors, and latency
- Link each tool call back to the reasoning step that triggered it
- Identify repeated tool failures or patterns of misuse
- Use tool logs to validate whether agent decisions were grounded

#### Diagnosing Failures Systematically

1. **Classify failure types** — distinguish tool failures, reasoning errors, and state corruption
2. **Reproduce failures reliably** — isolate inputs that trigger the failure consistently
3. **Trace root cause** — walk back through the trace to the earliest incorrect step
4. **Validate failure source** — confirm whether the bug is in the tool, the prompt, or the agent logic

### Demo: Retry and Fallback

This demo shows a primary tool that fails intentionally, a retry with clarified instructions, and an automatic fallback to a simpler tool when the primary continues to fail.

See [`04/demos.ipynb`](04/demos.ipynb) for the full notebook.

In [4]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Primary tool — always fails (simulated schema validation error)
def structured_json_summarizer(text: str) -> dict:
    raise RuntimeError("Schema validation failed")

# Fallback tool — always succeeds
def simple_text_summarizer(text: str) -> dict:
    return {"summary": text[:100], "confidence": "low"}

tools = [
    {"type": "function", "function": {
        "name": "structured_json_summarizer",
        "description": "Generate a structured JSON summary",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}
    }},
    {"type": "function", "function": {
        "name": "simple_text_summarizer",
        "description": "Generate a simple plain-text summary",
        "parameters": {"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]}
    }}
]

user_prompt = "Convert the following text into a structured JSON summary.\n\nText:\nAgentic AI systems autonomously plan and act using tools in multi-step workflows."

# Attempt 1 — model selects primary tool
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": user_prompt}],
    tools=tools, tool_choice="auto"
)
tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

try:
    result = structured_json_summarizer(**args)
except Exception as e:
    print(f"Primary tool failed: {e}")

    # Retry with simplified instructions
    retry_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "The previous attempt failed. Please use the simple fallback tool."},
            {"role": "user", "content": user_prompt}
        ],
        tools=tools, tool_choice={"type": "function", "function": {"name": "simple_text_summarizer"}}
    )
    retry_args = json.loads(retry_response.choices[0].message.tool_calls[0].function.arguments)
    fallback_result = simple_text_summarizer(**retry_args)
    print("Fallback tool output:")
    print(json.dumps(fallback_result, indent=2))

Primary tool failed: Schema validation failed
Fallback tool output:
{
  "summary": "Agentic AI systems autonomously plan and act using tools in multi-step workflows.",
  "confidence": "low"
}


## Module 05: Agent Orchestration and Best Practices

Agent orchestration is the coordination of AI agents, tools, and workflows to achieve complex, multi-step objectives. It defines how agents plan tasks, share context, manage state, and sequence actions — and how to build, monitor, and optimize these systems for production use.

### Overview of Agent Orchestration

Orchestration coordinates agents, tools, and data while enforcing correct execution order. It enables reliable step-by-step handoffs, prevents loops and partial execution, manages failures and retries, and improves overall system maintainability.

**Key principles of agent orchestration:**
- **Clear roles** — every agent and component has a defined responsibility
- **Task decomposition** — complex goals are split into manageable steps
- **Deterministic flow** — execution order is explicit and predictable
- **State management** — shared state is maintained and protected
- **Tool awareness** — orchestrator knows what tools are available and when to use them
- **Observability** — every step is visible, logged, and traceable

### Agent Frameworks

Agent frameworks are software libraries that provide reusable building blocks for creating, orchestrating, and managing autonomous agents that reason, use tools, maintain state, and execute multi-step workflows.

#### Major Frameworks and Their Use Cases

| Framework | Primary Use Case |
|---|---|
| **LangChain** | Tool-driven multi-step reasoning; broad ecosystem of integrations |
| **LlamaIndex** | Data-grounded retrieval and querying over structured and unstructured sources |
| **AutoGPT** | Autonomous task planning and execution with minimal human intervention |
| **CrewAI** | Role-based multi-agent collaboration with defined agent personas |

**Framework trade-offs:**

| Dimension | Consideration |
|---|---|
| **Complexity** | Frameworks add abstraction layers that can obscure agent behavior |
| **Performance** | Framework overhead may introduce latency vs. custom implementations |
| **Flexibility** | Frameworks constrain architecture to their design patterns |
| **Cost** | Extra API calls and tokens due to framework abstractions |

### Custom Agent Implementations

#### When to Build Custom Agents

Consider building custom agents instead of using a framework when:
- Frameworks constrain the architecture or execution flow you need
- You require full control over reasoning and tool use
- Strict security, auditability, or compliance requirements apply
- Complex business logic exceeds what framework abstractions can express
- You need to optimize for performance, latency, or cost

#### Foundational Components for Agent Workflows

Custom agent workflows need these four infrastructure components:

- **Conversation state** — the current context window and message history
- **Working memory** — scratchpad for intermediate results within a task
- **Task queues** — async execution and coordination of long-running steps
- **Result storage** — persistent store for completed step outputs

#### Strengths and Weaknesses of Custom Agents

| Strengths | Weaknesses |
|---|---|
| Full architectural control | Higher development effort |
| Tailored business logic | Longer initial delivery timelines |
| Seamless system integration | Increased maintenance burden |
| Minimal external dependencies | Limited ecosystem support |

#### Measuring and Optimizing Agent Performance

- **Task success rate** — define success criteria and measure pass rate
- **Steps to completion** — track how many iterations the agent takes; reduce unnecessary actions
- **Cost per task** — monitor token and API usage per task
- **Latency** — measure across planning, execution, and tool call phases separately
- **Output quality** — evaluate using automated scoring and human review

### Persistent State Management

#### Conversation State and Working Memory

Agents need to persist state across steps to maintain context, enable recovery, and manage memory eviction:

- **Persist history** — store the full conversation and action history
- **Maintain context** — keep relevant prior steps visible to the agent
- **Enable recovery** — checkpoint state so execution can resume after failure
- **Store intermediates** — save partial results for use in later steps
- **Separate memory** — keep long-term memory distinct from short-term working memory
- **Manage eviction** — remove stale or irrelevant context to stay within token limits

#### Task Queues and Result Storage

- Decouple planning and execution using task queues (Redis, Apache Kafka)
- Enable asynchronous long-running agent workflows that don't block
- Persist results in fast and durable storage
- Track progress with explicit task states (pending, running, complete, failed)

#### Handling Long-running Workflows

Long-running workflows require durable state, checkpointing, and recovery to safely pause and resume agent execution. Task queues and asynchronous execution help manage extended processing, retries, and partial failures.

### Agent Observability: Thoughts, Actions, and Tools

Full observability means capturing every dimension of agent execution:

| Dimension | What to capture |
|---|---|
| **Agent thoughts** | Internal reasoning steps and chain-of-thought traces |
| **Action execution** | What action was taken, with what inputs, and what outcome |
| **Tool call logging** | Tool name, inputs, outputs, latency, and errors |
| **Reasoning traceability** | Link each action back to the reasoning step that produced it |
| **Structured observability** | Consistent log schema for automated analysis and alerting |

### Demo: Observability, Debugging, and Performance

This demo builds a two-step LangChain pipeline (summarize → quiz), wraps each step with timing and step tracking, and computes KPIs including latency, steps to completion, and a cost estimate.

See [`05/demos.ipynb`](05/demos.ipynb) for the full notebook.

In [5]:
import os
import time
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=os.environ["OPENAI_API_KEY"])

metrics = {"steps": 0, "start_time": None, "end_time": None, "success": False}

summary_chain = (
    ChatPromptTemplate.from_messages([("system", "Summarize the text in one sentence."), ("user", "{text}")])
    | llm
    | RunnableLambda(lambda x: x.content)
)

quiz_chain = (
    ChatPromptTemplate.from_messages([("system", "Generate 3 quiz questions from the summary."), ("user", "{summary}")])
    | llm
    | RunnableLambda(lambda x: x.content)
)

metrics["start_time"] = time.time()

summary = summary_chain.invoke({"text": "Agentic AI systems can reason and act using tools."})
metrics["steps"] += 1
print("Summary:\n", summary)

quiz = quiz_chain.invoke({"summary": summary})
metrics["steps"] += 1
print("\nQuiz:\n", quiz)

metrics["end_time"] = time.time()
metrics["success"] = True

latency = metrics["end_time"] - metrics["start_time"]
print(f"\nKPIs")
print(f"Task Success:         {metrics['success']}")
print(f"Steps to Completion:  {metrics['steps']}")
print(f"Latency (seconds):    {round(latency, 2)}")
print(f"Estimated Cost:       Low (2 short LLM calls)")

Summary:
 Agentic AI systems possess the ability to reason and take actions by utilizing various tools.

Quiz:
 1. What is a key characteristic of agentic AI systems?
   a) They can only process data without taking actions.
   b) They possess the ability to reason and take actions using various tools.
   c) They are limited to performing simple tasks.
   d) They do not utilize any tools in their operations.

2. Which of the following best describes the functionality of agentic AI systems?
   a) They are designed solely for data storage.
   b) They can reason and execute actions based on their reasoning.
   c) They only operate under human supervision.
   d) They are incapable of using tools.

3. What enables agentic AI systems to perform tasks effectively?
   a) Their reliance on human input for every decision.
   b) Their ability to reason and utilize various tools.
   c) Their limited understanding of complex problems.
   d) Their focus on a single type of task.

KPIs
Task Success:  